# 2022 年韩国关税署 10 月 1–20 日原稿 HWP 来源

冻结官方页和HWP两次收取，先 NAS 备份，后用已验2021 HWP/PDF作二进制解析对照。无 SOXX 结果、拟合或账户。

In [1]:
import hashlib,json,subprocess,tempfile,sys,zlib
from datetime import datetime,timezone,date
from pathlib import Path
import olefile
from src.data.nas_market_data import publish_nas_snapshot,materialize_nas_snapshot
root=Path.cwd();card=root/'research/kcs_early20_2022_oct_original_hwp_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists() and olefile.__version__=='0.47'
prior=plan['prior_official_2021'];assert sha(root/prior['card']/'summary.json')==prior['summary_sha256'];prior_raw=root/'data/snapshots/kcs_early20_2021_oct_original_source_20261023'
assert sha(prior_raw/'2021_early20.hwp')==prior['HWP_sha256'] and sha(prior_raw/'2021_early20.pdf')==prior['PDF_sha256'] and json.loads((root/prior['card']/'nas_binding.json').read_text())['verified']
raw=root/'data/snapshots/kcs_early20_2022_oct_original_hwp_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','plan_sha256':sha(card/'plan.json'),'GET_attempts':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];failure=None
for spec in plan['exact_requests']:
    state['GET_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
    body=raw/spec['name'];headers=raw/(spec['name']+'.headers');started=datetime.now(timezone.utc).isoformat();t=plan['transport']
    cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(t['connect_timeout_seconds']),'--max-time',str(t['total_timeout_seconds_each']),'--max-filesize',str(t['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',t['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
    try:
        run=subprocess.run(cmd,capture_output=True,text=True,timeout=40);parts=run.stdout.split(' ',1);receipt={'name':spec['name'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':int(parts[0]) if parts[0].isdigit() else None,'final_url':parts[1] if len(parts)>1 else None,'transport_error':run.stderr[:250]}
    except Exception as exc:receipt={'name':spec['name'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
    receipt.update(body_sha256=sha(body) if body.exists() else None,body_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
    receipts.append(receipt);(raw/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
    if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists() or body.stat().st_size>t['max_response_bytes_each']:
        failure='transport_or_size:'+spec['name'];break
    data=body.read_bytes()
    if spec['name']=='official_index.html' and not all(z in data for z in (b'156531875',b'197006618',b'2022.10.21')):failure='index_identity';break
    if spec['name'].endswith('.hwp') and not data.startswith(bytes.fromhex('D0CF11E0A1B11AE1')):failure='HWP_signature';break
manifest={'schema_version':1,'experiment':plan['experiment'],'plan_sha256':state['plan_sha256'],'capture_utc':datetime.now(timezone.utc).isoformat(),'receipts':receipts,'failure':failure,'historical_first_public_bytes_qualified':False}
(raw/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n');(card/'source_receipt.json').write_text(json.dumps(manifest,indent=2)+'\n')
files={str(p.relative_to(root)):'raw' for p in raw.iterdir() if p.is_file()}
try:
    binding=publish_nas_snapshot(workspace_root=root,source_manifest=raw/'source_manifest.json',files=files,as_of=date.today().isoformat(),source_metadata={'experiment':plan['experiment'],'source':'Korea government 2022 Oct1-20 KCS HWP original'})
    (card/'nas_binding.json').write_text(json.dumps(binding,indent=2)+'\n')
    with tempfile.TemporaryDirectory(prefix='kcs-2022-cold-') as temp:
        readback=materialize_nas_snapshot(binding,workspace_root=Path(temp))
        for obj in binding['objects']:assert sha(Path(temp)/obj['local_path'])==obj['sha256']
    assert binding['verified'] and readback['restored_objects']==len(files)
except Exception as exc:
    failure=(failure+';' if failure else '')+'NAS:'+type(exc).__name__+':'+str(exc)[:180];binding=None
def section(path):
    with olefile.OleFileIO(path) as doc:
        assert doc.exists('BodyText/Section0') and doc.exists('PrvText')
        compressed=doc.openstream('BodyText/Section0').read();body=zlib.decompress(compressed,-15)
        preview=doc.openstream('PrvText').read().decode('utf-16le','replace')
        meta=doc.get_metadata()
        return body,preview,{'OLE_streams':['/'.join(x) for x in doc.listdir()],'created':str(meta.create_time) if meta.create_time else None,'last_saved':str(meta.last_saved_time) if meta.last_saved_time else None}
def positions(data,token):
    out=[]
    for encoding in ('utf-16le','utf-16be'):
        needle=token.encode(encoding);start=0
        while True:
            i=data.find(needle,start)
            if i<0:break
            out.append({'offset':i,'encoding':encoding});start=i+1
    return sorted(out,key=lambda x:x['offset'])
parsed=None
if not failure:
    try:
        control,_,control_meta=section(prior_raw/'2021_early20.hwp');current,preview,meta=section(raw/'2022_early20.hwp')
        def pair(data,label,amount,yoy):
            labels=positions(data,label);amounts=positions(data,amount);yoys=positions(data,yoy)
            matches=[{'label_offset':a['offset'],'amount_offset':b['offset'],'yoy_offset':c['offset'],'amount_encoding':b['encoding'],'yoy_encoding':c['encoding']} for a in labels for b in amounts for c in yoys if 0<b['offset']-a['offset']<=600 and 0<c['offset']-b['offset']<=500]
            return {'label_occurrences':len(labels),'amount_occurrences':len(amounts),'yoy_occurrences':len(yoys),'nearby_matches':matches}
        control_match=pair(control,'반도체','6,394','23.9');candidate_match=pair(current,'반도체','5,573','12.8')
        assert control_match['nearby_matches'] and candidate_match['nearby_matches']
        parsed={'parser':'olefile0.47 HWP5 BodyText/Section0 raw-deflate UTF-16LE/BE token-offset audit','2021_control':control_match,'2022':candidate_match,'2022_preview_semiconductor_yoy_seen':('반도체' in preview and '12.8' in preview),'2021_ole_metadata':control_meta,'2022_ole_metadata':meta,'2022_body_decompressed_bytes':len(current),'2022_preview_chars':len(preview),'2022_expected_secondary_transcription_not_source':'5,573 USD million and -12.8%'}
    except Exception as exc:failure='HWP_parse_or_control:'+type(exc).__name__+':'+str(exc)[:170]
state.update(status='completed' if not failure else 'stopped',failure=failure);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
result={'decision':'official_2022_KCS_HWP_semiconductor_amount_and_yoy_binary_corrob_source_only' if not failure else 'needs_2022_KCS_original_or_parser','failure':failure,'GET_attempts':state['GET_attempts'],'NAS_source_id':binding['source_id'] if binding else None,'NAS_readback_verified':bool(binding),'parsed':parsed,'historical_first_public_bytes_qualified':False,'cross_year_training_panel_qualified':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'strategy_forward_days':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n');print(json.dumps({k:result[k] for k in ('decision','failure','GET_attempts','NAS_readback_verified','parsed')}))


{"decision": "official_2022_KCS_HWP_semiconductor_amount_and_yoy_binary_corrob_source_only", "failure": null, "GET_attempts": 2, "NAS_readback_verified": true, "parsed": {"parser": "olefile0.47 HWP5 BodyText/Section0 raw-deflate UTF-16LE/BE token-offset audit", "2021_control": {"label_occurrences": 5, "amount_occurrences": 2, "yoy_occurrences": 4, "nearby_matches": [{"label_offset": 17420, "amount_offset": 17562, "yoy_offset": 17709, "amount_encoding": "utf-16be", "yoy_encoding": "utf-16be"}, {"label_offset": 17420, "amount_offset": 17562, "yoy_offset": 17710, "amount_encoding": "utf-16be", "yoy_encoding": "utf-16le"}, {"label_offset": 17420, "amount_offset": 17563, "yoy_offset": 17709, "amount_encoding": "utf-16le", "yoy_encoding": "utf-16be"}, {"label_offset": 17420, "amount_offset": 17563, "yoy_offset": 17710, "amount_encoding": "utf-16le", "yoy_encoding": "utf-16le"}]}, "2022": {"label_occurrences": 5, "amount_occurrences": 4, "yoy_occurrences": 2, "nearby_matches": [{"label_offset

# 2022 年韩国关税署 10 月 1–20 日半导体出口原稿

韩国政府政策门户 [2022-10-21 关税署原稿页](https://www.korea.kr/briefing/pressReleaseView.do?newsId=156531875)及唯一 HWP 附件均按冻结 URL 一次取得，原始响应、接收时间、SHA-256 已存入 Yanxuan NAS，远端哈希与定向冷恢复通过。HWP 预览标明 **2022-10-21 09:00 KST** 分发，可立即报道；这是文档自述时间，不是独立的首次在线收据。

正文 OLE 压缩段在“半导体”出口表格标签附近依次包含 **5,573 百万美元**和 **同比 −12.8%**。同一二进制读取方式先在已验的 2021 官方 HWP 上找到 **6,394、+23.9%**，与该年的政府 PDF 第 2 页目视结果一致。`5,573` 在 2022 文档里也指另一处不同量纲的全年累计总额，所以本卡只接受紧随半导体表格标签、且与 −12.8% 同行相邻的那个金额；精确字节偏移和复核方法见 `source_review.json`。跨年计算 `(5,573/6,394−1)×100 = −12.840%`，与一位小数的 −12.8% 一致；两年的工作日为 **13** 与 **13.5**，原始同比不能直接当成不受日历影响的需求惊喜。

决定：`official_2022_KCS_HWP_semiconductor_amount_and_yoy_binary_corrob_source_only`。这补上与 2021 同口径的一个官方历史实例，**仍不能证明今天收到的 HWP 字节在 2022-10-21 首次公开，也不是完整 2018–2022 原版训练面**。零 SOXX 结果读取、拟合、账户和策略前瞻日；完整 Stage1 0bp Tiered 对 SOXX 的硬门仍未通过。
